# Varn training on Kaggle

The one training notebook for Kaggle. Run the cells top to bottom (or **Save Version ▸ Save & Run All**).

- Reads only from `/kaggle/input`, writes only to `/kaggle/working`. No Google Drive, no Colab.
- Stops itself at `MAX_HOURS`, so a session never dies mid-epoch with nothing saved.
- Prints one status line per epoch. If the Logs tab stops moving, the run has stalled.
- Needs two private datasets attached under **Input**: your labeled clips (a YOLO dataset with a `data.yaml`) and the checkpoint dataset containing `last.pt`.

In [ ]:
# ---- Settings: the only cell you should need to edit ----
CLIPS_DATASET = "varn-clips"            # folder name under /kaggle/input (the dataset's slug)
CHECKPOINT_DATASET = "varn-checkpoint"  # folder name under /kaggle/input that contains last.pt
REQUIRE_CHECKPOINT = True               # True: stop if last.pt is missing. False: start from BASE_WEIGHTS
BASE_WEIGHTS = "yolov8n.pt"             # only used when REQUIRE_CHECKPOINT is False (needs internet)

EPOCHS = 100        # upper bound; MAX_HOURS usually ends the run first
IMGSZ = 960         # 960 for the final number, 800 as the fallback if memory or time is tight
BATCH = 16          # lower to 8 if you hit "CUDA out of memory"
MAX_HOURS = 8       # hard time cap for training (Kaggle sessions end at 9-12 h)

In [ ]:
# ---- Setup, GPU check, and fail-fast input checks ----
import csv, os, shutil, sys, time
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")


def stop(message):
    """End the notebook with a plain-English reason (no traceback noise)."""
    print("\nSTOPPED: " + message)
    raise SystemExit(message)


if not INPUT.exists():
    stop("/kaggle/input does not exist. This notebook only runs on Kaggle.")

try:
    import torch
except ImportError:
    stop("PyTorch is not installed in this session.")
print("torch", torch.__version__)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")
else:
    stop("No GPU found. Open Settings (right panel) > Accelerator and pick a GPU (T4 x2 or P100), then run again.")

try:
    import ultralytics
except ImportError:
    os.system(f"{sys.executable} -m pip install -q ultralytics")
    import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT
print("ultralytics", ultralytics.__version__)
if "time" not in DEFAULT_CFG_DICT:
    stop(f"Ultralytics {ultralytics.__version__} has no 'time' (hours) training argument, so the time cap cannot work. "
         "Run: pip install -U ultralytics, restart the session, and run again.")
print("time cap supported: yes")

attached = sorted(p.name for p in INPUT.iterdir())
print("Attached inputs:", attached or "none")


def find_input(name, what):
    path = INPUT / name
    if not path.is_dir():
        stop(f"Can't find the {what} '{name}' under /kaggle/input. Attached inputs are: {attached or 'none'}. "
             f"Attach it with Add Input in the right panel, or fix the name in the Settings cell.")
    return path


clips_dir = find_input(CLIPS_DATASET, "clips dataset")
yamls = sorted(clips_dir.rglob("data.yaml"))
if not yamls:
    stop(f"The clips dataset '{CLIPS_DATASET}' has no data.yaml. It needs a YOLO layout: images/ and labels/ folders plus data.yaml.")
src_yaml = yamls[0]

last_pt = None
if REQUIRE_CHECKPOINT:
    ckpt_dir = find_input(CHECKPOINT_DATASET, "checkpoint dataset")
    found = sorted(ckpt_dir.rglob("last.pt"))
    if not found:
        stop(f"The checkpoint dataset '{CHECKPOINT_DATASET}' has no last.pt. Upload it from the previous run's output, then attach it.")
    last_pt = found[0]
    print("Checkpoint:", last_pt, f"({last_pt.stat().st_size / 1e6:.1f} MB)")
else:
    print("No checkpoint required; starting from", BASE_WEIGHTS)
print("Dataset config:", src_yaml)

In [ ]:
# ---- Point data.yaml at the read-only input (inputs can't be written to) ----
import yaml

cfg = yaml.safe_load(src_yaml.read_text())
root = Path(cfg.get("path") or src_yaml.parent)
if not root.is_absolute() or not root.exists():
    root = src_yaml.parent / (cfg.get("path") or "")
cfg["path"] = str(root.resolve())
for key in ("train", "val"):
    if not cfg.get(key):
        stop(f"data.yaml has no '{key}' entry, so there is nothing to {'train on' if key == 'train' else 'score against'}.")
    if not (Path(cfg["path"]) / cfg[key]).exists():
        stop(f"data.yaml says {key}: {cfg[key]}, but {Path(cfg['path']) / cfg[key]} does not exist.")

data_yaml = WORK / "data.yaml"
data_yaml.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Classes:", cfg.get("names"))
print("Wrote", data_yaml)

# Ultralytics writes .cache files next to labels; that fails on a read-only input and only costs a warning.

In [ ]:
# ---- Train: one status line per epoch ----
from ultralytics import YOLO

RUNS = WORK / "runs"
RUN_NAME = "varn"
start = time.time()
state = {"epoch_start": time.time()}


def gpu_gb():
    return torch.cuda.memory_reserved() / 1e9 if torch.cuda.is_available() else 0.0


def on_epoch_end(trainer):
    now = time.time()
    m = trainer.metrics or {}
    print(f"[epoch {trainer.epoch + 1}/{trainer.epochs}] "
          f"elapsed {(now - start) / 60:.1f} min | epoch took {(now - state['epoch_start']) / 60:.1f} min | "
          f"mAP50 {m.get('metrics/mAP50(B)', float('nan')):.4f} | "
          f"mAP50-95 {m.get('metrics/mAP50-95(B)', float('nan')):.4f} | "
          f"GPU mem {gpu_gb():.1f} GB", flush=True)
    state["epoch_start"] = now


train_args = dict(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, time=MAX_HOURS,
                  project=str(RUNS), name=RUN_NAME, exist_ok=True, workers=2, amp=True, patience=20)

print(f"Training: imgsz={IMGSZ} batch={BATCH} epochs<={EPOCHS} time cap={MAX_HOURS} h", flush=True)
trainer_model = None
if last_pt is not None:
    local_last = WORK / "resume_last.pt"
    shutil.copyfile(last_pt, local_last)
    try:
        model = YOLO(str(local_last))
        model.add_callback("on_fit_epoch_end", on_epoch_end)
        print("Trying to resume the interrupted run from last.pt ...", flush=True)
        model.train(resume=True, imgsz=IMGSZ, batch=BATCH)
        trainer_model = model
    except Exception as exc:
        # A finished run's last.pt has no optimizer state, so it can't be resumed. Continue from its weights instead.
        print(f"Could not resume exactly ({type(exc).__name__}: {str(exc)[:200]}).")
        print("Continuing from last.pt's weights with a fresh optimizer instead.", flush=True)
        state["epoch_start"] = time.time()
        model = YOLO(str(local_last))
        model.add_callback("on_fit_epoch_end", on_epoch_end)
        model.train(**train_args)
        trainer_model = model
else:
    model = YOLO(BASE_WEIGHTS)
    model.add_callback("on_fit_epoch_end", on_epoch_end)
    model.train(**train_args)
    trainer_model = model
print(f"Training finished after {(time.time() - start) / 3600:.2f} h", flush=True)

In [ ]:
# ---- Save outputs, score, export ONNX ----
save_dir = Path(trainer_model.trainer.save_dir)
weights = save_dir / "weights"
best, last = weights / "best.pt", weights / "last.pt"
for f in (best, last):
    if not f.exists():
        stop(f"Training ended but {f} was not written. Look at the Logs above for the error.")

shutil.copyfile(last, WORK / "last.pt")
shutil.copyfile(best, WORK / "best.pt")
results_dir = WORK / "results"
if results_dir.exists():
    shutil.rmtree(results_dir)
shutil.copytree(save_dir, results_dir, ignore=shutil.ignore_patterns("weights"))

best_model = YOLO(str(best))
onnx_path = Path(best_model.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True))
shutil.copyfile(onnx_path, WORK / "best.onnx")
print("Saved to /kaggle/working:", sorted(p.name for p in WORK.iterdir() if p.is_file()))


def show(label, r):
    print(f"{label}: mAP50 {r.box.map50:.4f} | mAP50-95 {r.box.map:.4f} | precision {r.box.mp:.4f} | recall {r.box.mr:.4f}")
    for i, name in best_model.names.items():
        if i < len(r.box.ap50):
            print(f"    {name}: AP50 {r.box.ap50[i]:.4f}")


print("\n=== FINAL NUMBERS (best.pt) ===")
print(f"imgsz {IMGSZ}, batch {BATCH}, run folder {save_dir}")
show("validation", best_model.val(data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, split="val", verbose=False, plots=False))

test_rel = cfg.get("test")
if test_rel and (Path(cfg["path"]) / test_rel).exists():
    show("frozen test set", best_model.val(data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, split="test", verbose=False, plots=False))
else:
    print("frozen test set: none found (data.yaml has no 'test:' entry that exists), so nothing was scored on one. "
          "Add a frozen test split to the clips dataset and set 'test:' in its data.yaml to get this number.")

rows = list(csv.DictReader(open(save_dir / "results.csv"))) if (save_dir / "results.csv").exists() else []
if rows:
    print(f"epochs completed: {len(rows)}")